## Model development using Ensemble of efficientNet-B0 + ResNet50 with FastAI framework

This notebook presents an ensemble approach for paddy leaf disease classification using two deep learning architectures: EfficientNet-B0 and ResNet50. Both models were fine-tuned independently using the FastAI framework with test-time augmentation (TTA) and tracked via Weights & Biases. Their predictions were then averaged (soft voting) to form a robust ensemble classifier. The final model achieved a 97.52% accuracy on the test set. Evaluation through a classification report and confusion matrix confirmed strong generalization across all disease classes. Further improvements may be achieved by increasing the number of training epochs or incorporating additional regularization and augmentation strategies.

<!-- ####  a. Import dataset

1. Download the Paddy Doctor split balanced dataset file ``paddy-doctor-diseases-small-400-split.zip`` from [this link](https://ieee-dataport.org/documents/paddy-doctor-visual-image-dataset-automated-paddy-disease-classification-and-benchmarking)
2. Create a new folder named ``paddy-doctor`` in your Google Drive's root directory and upload the ``paddy-doctor-diseases-small-400-split.zip`` file there. -->

Environment Setup and Data Loading for FastAI-based Training
-------

In [ ]:
# Import glob for file operations
import glob

# Try importing fastkaggle, install if not already available
try:
    import fastkaggle
except ModuleNotFoundError:
    !pip install -Uq fastkaggle

# Import key modules from fastkaggle and fastai
from fastkaggle import *
from fastai.vision.all import *

# Set random seed for reproducibility
set_seed(42)

# Define path to the competition dataset (local or Kaggle format)
competition = '/kaggle/input/paddy-doctor-disease/Augmented and split - 26000 augmented images split into train (80) sets'

# Use fastkaggle helper to set up the dataset and environment
# Also installs fastai and timm (PyTorch image models)
path = setup_comp(competition, install='fastai "timm>=0.6.2.dev0"')
print(path)

# Define path and load all image files for training
train_path = path / 'train'
train_files = get_image_files(train_path)

# Define path and load all image files for testing (sorted for consistency)
test_path = path / 'test'
test_files = get_image_files(test_path).sorted()

# Load training labels from metadata CSV file
train_df = pd.read_csv(path / 'metadata-train.csv')
print(train_df.shape)

# Display class distribution in the training dataset
train_df.label.value_counts()

DataBlock Definition and DataLoaders Creation with Augmentations
----------

In [ ]:
# Define a DataBlock for image classification using FastAI
dblock = DataBlock(
    blocks=(ImageBlock, CategoryBlock),       # Define the input (images) and target (categories) types
    get_items=get_image_files,                # Function to retrieve image file paths
    get_y=parent_label,                       # Use the parent folder name as the label
    splitter=RandomSplitter(valid_pct=0.2, seed=42),  # Randomly split 20% of data for validation
    item_tfms=Resize(480, method='squish'),   # Resize all images to 480x480 using 'squish' to fit without padding
    batch_tfms=aug_transforms(                # Apply data augmentations on batches
        size=224,                             # Final image size after crop
        min_scale=0.75,                       # Minimum scale for cropping
        max_rotate=20,                        # Allow rotation up to 20 degrees
        max_zoom=1.2,                         # Allow zoom-in up to 1.2x
        max_warp=0.2,                         # Apply perspective warping
        p_affine=0.75,                        # 75% chance of applying affine transforms
        p_lighting=0.75                       # 75% chance of applying lighting transforms
    ) + [RandomErasing(p=0.5, max_count=1)]   # Randomly erase one region in 50% of the images for robustness
)

# Create the DataLoaders (train and valid loaders)
dls = dblock.dataloaders(train_path)

Quick DataLoaders Creation Using ImageDataLoaders.from_folder
----------

In [ ]:
# Create ImageDataLoaders directly from folders using FastAI's high-level API
dls = ImageDataLoaders.from_folder(
    train_path,                       # Root directory with class folders
    valid_pct=0.2,                    # Use 20% of data for validation
    seed=42,                          # Seed for reproducible split
    item_tfms=Resize(480, method='squish'),  # Resize all images to 480x480 using 'squish' (no padding)
    batch_tfms=aug_transforms(        # Apply default data augmentations on batches
        size=224,                     # Crop to 224x224 after resizing
        min_scale=0.75               # Allow scaling down to 75% before cropping
    )
)

Weights & Biases (wandb) Initialization for Ensemble(efficientNet-B0 + ResNet50) Training Run
--------------

In [ ]:
# Import Weights & Biases for experiment tracking
import wandb

# Log in to Weights & Biases using an API key
wandb.login(key="")

# Initialize a new W&B run for the Ensemble(efficientNet-B0 + ResNet50) experiment
run = wandb.init(
    project="project-ablations",     # Project name on wandb dashboard
    name="Ensemble(efficientNet-B0 + ResNet50)",             # Run name to identify this experiment
    config={                         # Hyperparameter configuration
        "epochs": 20,                 # Number of training epochs
        "base_lr": 0.005,             # Base learning rate
        "weight_decay": 0.01,         # L2 regularization value
        "architecture": "Ensemble(efficientNet-B0 + ResNet50)"  # Model architecture name
    },
    reinit=True                      # Allow reinitializing this run if rerun in the same process
)

Creating and Configuring EfficientNet Learner with FastAI and W&B
---------

In [ ]:
# Import model creation utility and FastAI components
from timm import create_model
from fastai.vision.all import vision_learner, error_rate
from fastai.callback.wandb import WandbCallback

# -----------------------------
# EfficientNet Learner
# -----------------------------

# Define a wrapper to create an EfficientNet-B0 model using the timm library
def efficientnet_model(**kwargs):
    kwargs.pop('pretrained', None)  # Remove any externally passed 'pretrained' to avoid conflict
    return create_model("tf_efficientnet_b0_ns", pretrained=True, **kwargs)

# Create a FastAI vision learner using the EfficientNet backbone
learn_eff = vision_learner(
    dls,                          # DataLoaders object with training/validation sets
    efficientnet_model,          # Custom EfficientNet model function
    metrics=error_rate,          # Evaluation metric to monitor during training
    cbs=[
        MixUp(),                 # Data augmentation callback (MixUp for regularization)
        SaveModelCallback(       # Save best model based on error rate
            fname='eff_best_model',
            monitor='error_rate'
        ),
        WandbCallback(log_model=True)  # Log metrics and model checkpoints to wandb
    ]
).to_fp16()                       # Convert model to 16-bit floating point (mixed precision training)

# Set directory where model checkpoints will be saved
learn_eff.model_dir = "/kaggle/working/models"

ResNet50 Learner Setup for Ensemble Training
---------

In [ ]:
# -----------------------------
# ResNet50 Learner
# -----------------------------

# Define a wrapper to create a ResNet50 model using the timm library
def resnet_model(**kwargs):
    kwargs.pop('pretrained', None)  # Remove 'pretrained' to avoid conflict
    return create_model("resnet50", pretrained=True, **kwargs)

# Initialize a FastAI vision learner using ResNet50 backbone
learn_resnet = vision_learner(
    dls,                             # DataLoaders
    resnet_model,                   # ResNet50 model function
    metrics=[accuracy, error_rate], # Metrics to monitor during training
    cbs=[
        # MixUp(),                   # (Optional) MixUp for regularization
        SaveModelCallback(          # Save best model based on error rate
            fname='resnet_best_model',
            monitor='error_rate'
        ),
        WandbCallback(log_model=True)  # Log training to Weights & Biases
    ]
).to_fp16()                          # Use mixed precision for efficiency

# Set directory to save model weights
learn_resnet.model_dir = "/kaggle/working/models"


Fine-Tuning EfficientNet-B0 for Ensemble
----

In [ ]:
# Fine-tune the EfficientNet-B0 model for 40 epochs
# Using a base learning rate of 0.01 and weight decay of 0.01
learn_eff.fine_tune(40, base_lr=0.01, wd=0.01)

Fine-Tuning ResNet50 for Ensemble
--------

In [ ]:
# Fine-tune the ResNet50 model for 40 epochs
# Using a base learning rate of 0.01 and weight decay of 0.01
learn_resnet.fine_tune(40, base_lr=0.01, wd=0.01)

Exporting Trained Models and Logging to Weights & Biases
-----

In [ ]:
# Export the fine-tuned EfficientNet-B0 and ResNet50 models
learn_eff.export("eff_final_model.pkl")
learn_resnet.export("resnet_final_model.pkl")

# Log both exported models to Weights & Biases for versioning and reproducibility
wandb.save("eff_final_model.pkl")
wandb.save("resnet_final_model.pkl")

Validation Ensemble: Averaging Predictions from EfficientNet-B0 and ResNet50
----------

In [ ]:
# Get validation set predictions using Test-Time Augmentation (TTA) for both models
probs_eff, target = learn_eff.tta(dl=dls.valid)      # EfficientNet-B0 predictions and true labels
probs_resnet, _ = learn_resnet.tta(dl=dls.valid)     # ResNet50 predictions (ignore target, already captured)

# Perform soft-voting ensemble by averaging probabilities from both models
ensemble_probs = (probs_eff + probs_resnet) / 2

# Evaluate the ensemble performance using FastAI's error_rate metric
print("Ensembled Error Rate (Validation):", error_rate(ensemble_probs, target))

Logging Ensemble Validation Error to Weights & Biases
--------

In [ ]:
# Compute the ensemble error rate on the validation set
ensemble_val_error = error_rate(ensemble_probs, target)

# Log the validation error rate of the ensemble model to W&B
wandb.log({"ensemble_val_error_rate": ensemble_val_error})

Test Set Ensemble Predictions from EfficientNet-B0 and ResNet50
-----------------

In [ ]:
# Define test path and retrieve test image files in sorted order
test_path = path / 'test'
test_files = get_image_files(test_path).sorted()

# Extract true test class names from the folder structure (if available)
test_classes = [f.parent.name for f in test_files]

# Generate predictions on the test set using TTA for both models
probs_eff_test, _ = learn_eff.tta(dl=dls.test_dl(test_files))
probs_resnet_test, _ = learn_resnet.tta(dl=dls.test_dl(test_files))

# Average the predicted probabilities from both models (soft voting ensemble)
ensemble_test_probs = (probs_eff_test + probs_resnet_test) / 2

# Determine the predicted class index for each test image
preds = ensemble_test_probs.argmax(dim=1)

# Map predicted indices to actual class names using the dataloader's vocabulary
pred_classes = dls.vocab[preds]

**Results**

Ensemble Evaluation on Test Set: Accuracy and Classification Report
--------

In [ ]:
# Import evaluation and visualization libraries
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
import seaborn as sns

# Generate a classification report showing precision, recall, and F1-score for each class
cls_report = classification_report(
    test_classes,          # True labels (from folder structure)
    pred_classes,          # Predicted labels (from ensemble)
    digits=5               # Display metrics with 5 decimal places
)
print(cls_report)

# Calculate and print the overall accuracy of the ensemble on the test set
acc = accuracy_score(test_classes, pred_classes)
print(f"Ensemble Accuracy on Test Set: {acc:.5f}")

Logging Ensemble Test Accuracy to Weights & Biases
--------

In [ ]:
# Log the ensemble model's test accuracy to Weights & Biases for tracking
wandb.log({"ensemble_test_accuracy": acc})

Confusion Matrix Heatmap for Ensemble Predictions
----------

In [ ]:
# Import seaborn and confusion matrix function
import seaborn as sns
from sklearn.metrics import confusion_matrix

# Define a reusable function to plot confusion matrix heatmaps
def plot_heatmap(y_true, y_pred, class_names, ax, title):
    cm = confusion_matrix(y_true, y_pred)   # Compute confusion matrix
    sns.heatmap(
        cm, 
        annot=True,                         # Display counts in each cell
        square=True,                        # Use square cells
        xticklabels=class_names,           # Show class names on x-axis
        yticklabels=class_names,           # Show class names on y-axis
        fmt='d',                            # Format values as integers
        cmap=plt.cm.Blues,                 # Blue color map
        cbar=False,                        # Disable color bar
        ax=ax                              # Plot on provided axis
    )
    # Set font sizes and rotation for better readability
    ax.set_xticklabels(ax.get_xticklabels(), fontsize=12, rotation=45, ha="right")
    ax.set_yticklabels(ax.get_yticklabels(), fontsize=12)
    ax.set_ylabel('True Label', fontsize=12)
    ax.set_xlabel('Predicted Label', fontsize=12)

# Create a single subplot to visualize confusion matrix
fig, ax = plt.subplots(1, 1, figsize=(8, 6))

# Plot the confusion matrix for ensemble predictions
plot_heatmap(test_classes, pred_classes, dls.vocab, ax, title="Ensembled EfficientNet + ResNet50")

# Display the plot
plt.show()

Saving Ensemble Predictions to CSV
---------

In [ ]:
# Create a DataFrame containing the true and predicted labels for the test set
res = pd.DataFrame({
    "y_true": test_classes,   # Actual class labels
    "y_pred": pred_classes    # Predicted labels from the ensemble model
})

# Save the predictions to a CSV file for reporting or further analysis
res.to_csv('result.csv', index=False)

# Display the result DataFrame
res